In [ ]:
from src.utils.helper import extract_rephrase_from_jsonl
from src.utils.ai_question_generator import AIQuestionGenerator
from prompt_config.config import user_prompt_rephrase
import os
import pandas as pd

API_KEY = os.getenv('ANTHROPIC_API_KEY')

In [ ]:
# Generation of rephrase base
df = pd.read_csv("./data/processed/temporal_knowledge_wiki_merged_filtered_pageviews.csv")
df.head()

df = df[['id', 'question']].drop_duplicates()

## Apply manual question fix for original questions that didn't meet the criteria.
question_fix = pd.read_csv("./data/final/fix_questions.csv")
question_fix.head()


In [ ]:
# Left join question_fix onto df using question column
df = pd.merge(df, question_fix[['question', 'new_question']], on='question', how='left')
df = df.drop(columns=['id'])
# Reset index and create id column
df = df.reset_index(drop=True)
df['id'] = df.index

# # Replace original question with fixed question where available
# df['new_question'] = df['new_question'].fillna(df['question'])

# # Drop the new_question column since we no longer need it
# # df = df.drop('new_question', axis=1)
df.to_csv('./data/processed/anthropic_rephrase_base_recreated.csv', index=False)


In [ ]:
df = pd.read_csv('./data/processed/anthropic_rephrase_base_recreated.csv')
df.count()

In [ ]:
# System and user prompts for temporal questions
# from prompt_config.config import system_prompt, user_prompt


df = pd.read_csv('./data/processed/anthropic_rephrase_base_recreated.csv')

# Create new id column as row number starting from 1
df['id'] = df.index
df['id'] = df['id'].astype(str)


# df = df.head(10)

generator = AIQuestionGenerator(provider="anthropic"
                                , api_key=API_KEY)

# Prepare content for batch processing
batch_contents = []
for idx, row in df.iterrows():
    content_item = {
        'content': row['new_question'],
        'custom_id': row['id']  # Use URL as custom_id
    }
    batch_contents.append(content_item)


print(f"Creating batch with {len(batch_contents)} requests...")

# Create the batch -- Uncomment when required.
batch_info = generator.generate_qa_batch(
    contents=batch_contents,
    system_prompt="",
    user_prompt=user_prompt_rephrase,
    max_tokens=64000,
    num_questions=None
)








In [ ]:
df_rephrased = extract_rephrase_from_jsonl('./data/processed/anthropic_rephrased_results_2.jsonl')
df_rephrased.head()
# # Create new id column as row number starting from 1
# df['id'] = df.index + 1
# df['id'] = df['id'].astype(str)s

In [ ]:
# Read the base questions file
df_base = pd.read_csv('./data/processed/anthropic_rephrase_base_recreated.csv')
df_base['id'] = df_base.index
df_base['id'] = df_base['id'].astype(str)

# Join with rephrased questions, keeping only the rephrased_question column
df_base = df_base.merge(
    df_rephrased[['id', 'rephrased_question']], 
    on='id',
    how='left'
)
df_base

In [ ]:
# Read the parquet file containing temporal knowledge data
df_temporal = pd.read_parquet('./data/processed/temporal_knowledge_wiki_merged_filtered_pageviews.parquet')

# Join the temporal knowledge data with rephrased questions
# Subtract 1 from id in df_rephrased before merging

df_merged = df_temporal.merge(df_base, on='question', how='left')
df_merged = df_merged.drop_duplicates(subset=['new_question', 'event_year'])
df_merged.count()
df_merged.head()




In [ ]:
df_merged['relative_question'] = df_merged['rephrased_question']
df_merged['id'] = df_merged['id_x']
df_merged.drop(['rephrased_question', 'id_x', 'id_y'], inplace=True, axis=1)


In [ ]:
df_merged.count()

In [ ]:
def clean_relative_questions(df_input: pd.DataFrame) -> pd.DataFrame:
    """Remove the 'Rephrase:' prefix from relative_question values when present."""
    if "relative_question" not in df_input.columns:
        raise KeyError("Column 'relative_question' not found.")

    df_output = df_input.copy()
    before = df_output["relative_question"].astype(str)
    df_output["relative_question"] = (
        before
        .str.replace("Rephrase:", "", regex=False)
        .str.strip()
    )

    changed_rows = (before != df_output["relative_question"].astype(str)).sum()
    print(f"Updated {changed_rows:,} rows in 'relative_question'.")
    return df_output


# Keep this in the same pipeline style as the rest of the notebook.
df_merged = clean_relative_questions(df_merged)
display(df_merged[["relative_question"]].head())


In [ ]:
df_merged.to_parquet('./data/final/temporal_knowledge_wiki_merged_filtered_pageviews_rephrased.parquet', index=False)